# **SLM FINE TUNING**

In this notebook, we will fine-tune a pre-trained SLM (Small Language Model) on a specific dataset. The SLM is designed to understand and generate human language, making it ideal for various natural language processing tasks.

The SLM that we will fine-tune is `Qwen2.5-1.5B-Instruct`, which is a powerful language model with 1.5 billion parameters. This model has been pre-trained on a large corpus of text data and can be fine-tuned for specific tasks such as text classification, sentiment analysis, or question-answering.

In this case, we will use our own dataset that we built in `code_dataset.ipynb`. This dataset consists of text data of code snippets, completions, and comments. We will fine-tune the SLM on this dataset to improve its performance in understanding and generating code-related text.

For the fine-tunning process, we will use `LoRA` (Low-Rank Adaptation), which is a technique that allows us to efficiently fine-tune large language models by adapting only a small subset of the model's parameters. This approach significantly reduces the computational resources required for fine-tuning while still achieving good performance.

## **Installation of necessary packages**

In [1]:
%pip install "huggingface-hub>=0.34.0,<2.0" transformers==4.46.2 peft==0.13.2 accelerate==1.1.1 trl==0.12.1 "bitsandbytes==0.45.3" datasets==3.1.0 safetensors==0.4.5 --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 115.9 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.2/333.2 kB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.9/310.9 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 12.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 434.8/434.8 kB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 106.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are 

## **Importing Libraries**

In [2]:
import os
import torch
from datasets import load_dataset 
from peft import get_peft_model, LoraConfig, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

### **HUGGING FACE LOGIN**

In [3]:
from huggingface_hub import login

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = os.environ.get('HF_TOKEN')

if hf_token:
    login(token=hf_token)
else:
    login()  

## **FINE-TUNNING PREPARATION**

First, we need to set the configuration of `BitsAndBytesConfig` to use 4-bit quantization for the model. This will help reduce the memory footprint of the model during fine-tuning.

In [4]:
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
COMPUTE_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'} | bf16={USE_BF16} | compute_dtype={COMPUTE_DTYPE}")

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,  
)

GPU: Tesla T4 | bf16=True | compute_dtype=torch.bfloat16


Loading the pre-trained SLM model and tokenizer is the next step. We will use the `AutoModelForCausalLM` class of Hugging Face's transformers library.

In [5]:
repo_id = 'Qwen/Qwen2.5-1.5B-Instruct'
model = AutoModelForCausalLM.from_pretrained(
    repo_id,
    device_map='cuda:0',
    quantization_config=bnb_config,
    torch_dtype=COMPUTE_DTYPE,          
    attn_implementation='sdpa',          
)
model.config.use_cache = False    

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Let's see the weight of the model in memory before fine-tuning.

In [6]:
print(model.get_memory_footprint()/(1024**3))

1.045076608657837


Architecture of the model:

In [7]:
model

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2SdpaAttention(
          (q_proj): Linear4bit(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear4bit(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear4bit(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear4bit(in_features=1536, out_features=1536, bias=False)
          (rotary_emb): Qwen2RotaryEmbedding()
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear4bit(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear4bit(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)


Now, we can prepare the model for fine-tuning by using the `prepare_model_for_kbit_training` function from the `peft` library. This function will prepare the model for training with low-bit quantization.

We use the `LoraConfig` class to set up the configuration for LoRA fine-tuning. We specify the rank, alpha, dropout, and target modules for the LoRA adaptation.

In [8]:
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

config = LoraConfig(
    r=8, 
    lora_alpha=16, 
    bias='none',
    lora_dropout=0.05,
    task_type='CAUSAL_LM',
    target_modules=['o_proj', 'q_proj', 'k_proj', 'v_proj', 'gate_proj', 'up_proj' ,'down_proj']
)

model = get_peft_model(model, config)
model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2SdpaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lo

Let's see the new weight of the model in memory after preparing it for fine-tuning.

In [9]:
print(model.get_memory_footprint()/1e6)

1626.109184


Exploring the trainable parameters of the model after preparation for fine-tuning.

In [10]:
train_p, tot_p = model.get_nb_trainable_parameters()
print(f'Trainable parameters:      {train_p/1e6:.2f}M')
print(f'Total parameters:          {tot_p/1e6:.2f}M')
print(f'Percentage of trainable parameters: {100*train_p/tot_p:.2f}%')

Trainable parameters:      9.23M
Total parameters:          1552.95M
Percentage of trainable parameters: 0.59%


## **PREPARING THE DATASET FOR FINE-TUNING**

In [11]:
train_dataset = load_dataset("Juanxxo/smallcoder-dataset", split='train')
eval_dataset = load_dataset("Juanxxo/smallcoder-dataset", split='validation')

README.md:   0%|          | 0.00/643 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/6.27M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/768k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/374k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15250 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1794 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/897 [00:00<?, ? examples/s]

In [12]:
train_dataset

Dataset({
    features: ['input_code', 'target_completion', 'task', 'lang'],
    num_rows: 15250
})

In [13]:
eval_dataset

Dataset({
    features: ['input_code', 'target_completion', 'task', 'lang'],
    num_rows: 1794
})

Let's rename and delete some columns in the dataset to make it suitable for fine-tuning. We will rename the `input_code` column to `prompt` and the `target_completion` column to `completion`. We will also drop the `task` and `lang` column as it is not needed for fine-tuning.

In [14]:
train_dataset = train_dataset.rename_column("input_code", "prompt")
train_dataset = train_dataset.rename_column("target_completion", "completion")
train_dataset = train_dataset.remove_columns(["task", "lang"])
train_dataset

Dataset({
    features: ['prompt', 'completion'],
    num_rows: 15250
})

In [15]:
eval_dataset = eval_dataset.rename_column("input_code", "prompt")
eval_dataset = eval_dataset.rename_column("target_completion", "completion")
eval_dataset = eval_dataset.remove_columns(["task", "lang"])
eval_dataset

Dataset({
    features: ['prompt', 'completion'],
    num_rows: 1794
})

Printing the first few rows of the dataset to verify the changes.

In [16]:
train_dataset[0]

{'prompt': 'complete go: func (x *DiagnosticReport) UnmarshalJSON(data []byte) (err error) {\n\tx2 := diagnosticReport{}\n\tif err = json.Unmarshal(data, &x2); err == nil {\n\t\tif x2.Contained != nil {\n\t\t\tfor i := range x2.Contained {\n\t\t\t\tx2.Contained[i] = MapToResource(x2.Contained[i], true)',
 'completion': '\t\t\t}\n\t\t}\n\t\t*x = DiagnosticReport(x2)\n\t\treturn x.checkResourceType()\n\t}\n\treturn\n}'}

Create a function to preprocess the dataset for fine-tuning. This function will load the dataset with the expected format.

In [17]:
def format_dataset(examples):
  converted_sample = [
      {"role": "user", "content": examples["prompt"]},
      {"role": "assistant", "content": examples["completion"]},
  ]
  return {'messages': converted_sample}

In [18]:
dataset = train_dataset.map(format_dataset).remove_columns(['prompt', 'completion'])
dataset[1]['messages']

Map:   0%|          | 0/15250 [00:00<?, ? examples/s]

[{'content': 'complete java: public void setupCellPositoin(final CellPosition address) {\r\n        ArgUtils.notNull(address, "address");\r',
  'role': 'user'},
 {'content': '        setupCellPositoin(address.getRow(), address.getColumn());\r\n    }',
  'role': 'assistant'}]

In [19]:
eval_dataset = eval_dataset.map(format_dataset).remove_columns(['prompt', 'completion'])
eval_dataset[1]['messages']

Map:   0%|          | 0/1794 [00:00<?, ? examples/s]

[{'content': "complete javascript: function configureExecutor(executor) {\n  executor.defineCommand(\n      ExtensionCommand.GET_CONTEXT,\n      'GET',\n      '/session/:sessionId/moz/context');\n\n  executor.defineCommand(\n      ExtensionCommand.SET_CONTEXT,\n      'POST',",
  'role': 'user'},
 {'content': "      '/session/:sessionId/moz/context');\n\n  executor.defineCommand(\n      ExtensionCommand.INSTALL_ADDON,\n      'POST',\n      '/session/:sessionId/moz/addon/install');\n\n  executor.defineCommand(\n      ExtensionCommand.UNINSTALL_ADDON,\n      'POST',\n      '/session/:sessionId/moz/addon/uninstall');\n}",
  'role': 'assistant'}]

We need the model's tokenizer to preprocess the dataset. We will use the `AutoTokenizer` class from Hugging Face's transformers library to load the tokenizer for our pre-trained SLM model.

In [20]:
tokenizer = AutoTokenizer.from_pretrained(repo_id)
print(tokenizer.chat_template)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

{%- if tools %}
    {{- '<|im_start|>system\n' }}
    {%- if messages[0]['role'] == 'system' %}
        {{- messages[0]['content'] }}
    {%- else %}
        {{- 'You are Qwen, created by Alibaba Cloud. You are a helpful assistant.' }}
    {%- endif %}
    {{- "\n\n# Tools\n\nYou may call one or more functions to assist with the user query.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n<tools>" }}
    {%- for tool in tools %}
        {{- "\n" }}
        {{- tool | tojson }}
    {%- endfor %}
    {{- "\n</tools>\n\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\n<tool_call>\n{\"name\": <function-name>, \"arguments\": <args-json-object>}\n</tool_call><|im_end|>\n" }}
{%- else %}
    {%- if messages[0]['role'] == 'system' %}
        {{- '<|im_start|>system\n' + messages[0]['content'] + '<|im_end|>\n' }}
    {%- else %}
        {{- '<|im_start|>system\nYou are Qwen, created by Alibaba C

Let's apply the tokenizer to one example from the dataset to see how it works.

In [21]:
print(tokenizer.apply_chat_template(conversation=dataset['messages'][50], tokenize=False))

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
generate docstring go: func (s *DeleteClusterParameterGroupInput) SetParameterGroupName(v string) *DeleteClusterParameterGroupInput {
	s.ParameterGroupName = &v
	return s
}<|im_end|>
<|im_start|>assistant
// SetParameterGroupName sets the ParameterGroupName field's value.<|im_end|>



Pad the tokenized inputs and labels to the maximum length of the model's input. This will ensure that all inputs and labels have the same length, which is necessary for training the model.

In [22]:
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.eos_token_id

### **FINE-TUNNING EXECUTION**

In [23]:
sft = SFTConfig(
    gradient_checkpointing=True, 
    gradient_checkpointing_kwargs={"use_reentrant": False}, 
    gradient_accumulation_steps=1, 
    per_device_train_batch_size=4,
    auto_find_batch_size=True, 

    max_seq_length=256,
    packing=True,

    num_train_epochs=3,
    learning_rate=1e-4,

    optim='paged_adamw_8bit',

    evaluation_strategy = "epoch",
    save_strategy = "epoch",
    logging_steps=100,
    fp16 = True,
    logging_dir='./logs', 
    output_dir='./qwen2.5-1.5B-code-adapter',
    report_to='none',
    load_best_model_at_end=True
)

/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Mounting the Google Drive to save the fine-tuned model and training logs.

In [24]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Define the `SFTTrainer` class, which is a custom trainer for supervised fine-tuning. This class will handle the training loop, evaluation, and saving of the model.

In [25]:
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    args=sft,
    train_dataset=dataset,
    eval_dataset=eval_dataset
)

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

In [26]:
dl = trainer.get_train_dataloader()
batch = next(iter(dl))

In [27]:
len(batch['input_ids'][0]), len(batch['labels'][0])

(256, 256)

Executing the fine-tuning (the first part).

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,1.116300,1.099668


: 

: 

## Resuming Training from Checkpoint

During `SFTTrainer` training, Colab sessions can be unexpectedly interrupted.
To prevent losing progress, checkpoints are saved to Google Drive at the end
of each epoch using `save_strategy="epoch"`.

The `torch.load` patch is required due to a breaking change introduced in
**PyTorch 2.6**, which changed the default value of `weights_only` from
`False` to `True` for security reasons. Without this patch, the trainer
throws an `UnpicklingError` when attempting to restore the optimizer state
(`optimizer.pt`) saved with earlier versions of PyTorch.

In [28]:
import torch
original_load = torch.load

def patched_load(*args, **kwargs):
    kwargs['weights_only'] = False
    return original_load(*args, **kwargs)

torch.load = patched_load

In [29]:
trainer.train(
    resume_from_checkpoint="/content/drive/MyDrive/checkpoint-6000"
)

Epoch,Training Loss,Validation Loss
2,0.796200,0.955027
3,0.730200,0.883743


TrainOutput(global_step=10095, training_loss=0.3098495302016101, metrics={'train_runtime': 6236.6975, 'train_samples_per_second': 6.474, 'train_steps_per_second': 1.619, 'total_flos': 8.183261180775629e+16, 'train_loss': 0.3098495302016101, 'epoch': 3.0})

## **TESTING THE FINE-TUNED MODEL**

Create a function to encode the input prompt using the tokenizer.

In [30]:
def encode_prompt(tokenizer, sentence):
  sample = [{'role': 'user', 'content': sentence}]
  prompt = tokenizer.apply_chat_template(conversation=sample, tokenize=False, add_generation_prompt=True)
  return prompt

In [31]:
sentence = 'complete javascript: function gestureMove(ev) { \n if (!pointer || !typesMatch(ev, pointer)) return;'
prompt = encode_prompt(tokenizer, sentence)
print(prompt)

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
complete javascript: function gestureMove(ev) { 
 if (!pointer || !typesMatch(ev, pointer)) return;<|im_end|>
<|im_start|>assistant



In [32]:
sentence = """generate docstring go: func (n *EndpointSelector) IsWildcard() bool {
return n.LabelSelector != nil &&
len(n.LabelSelector.MatchLabels)+len(n.LabelSelector.MatchExpressions) == 0
}
"""

prompt2 = encode_prompt(tokenizer, sentence)
print(prompt2)

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
generate docstring go: func (n *EndpointSelector) IsWildcard() bool {
return n.LabelSelector != nil &&
len(n.LabelSelector.MatchLabels)+len(n.LabelSelector.MatchExpressions) == 0
}
<|im_end|>
<|im_start|>assistant



Create a function to execute the inference using the fine-tuned model. This function will take an input prompt, encode it using the tokenizer, and generate a response from the model.

In [33]:
def inference(model, tokenizer, prompt, max_new_tokens=64, skip_special_tokens=False):
  tokenized_input = tokenizer(
      prompt, return_tensors='pt', add_special_tokens=False
  ).to(model.device)

  model.eval() 

  gen_output = model.generate(**tokenized_input,
                              eos_token_id=tokenizer.eos_token_id,
                              max_new_tokens=max_new_tokens)

  output = tokenizer.batch_decode(gen_output, skip_special_tokens=skip_special_tokens)

  return output[0]

In [34]:
print(inference(model, tokenizer, prompt))

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
complete javascript: function gestureMove(ev) { 
 if (!pointer || !typesMatch(ev, pointer)) return;<|im_end|>
<|im_start|>assistant
 const { pointerId, type } = pointer;
 let { x : xGlobal, y: yGlobal } = ev;

 // We want to place the y-axis along the top edge of the screen.
 const scale = Math.max(1, ((ev.x - pointer.width / 2 + canvas.width/2)


In [35]:
print(inference(model, tokenizer, prompt2))

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
generate docstring go: func (n *EndpointSelector) IsWildcard() bool {
return n.LabelSelector != nil &&
len(n.LabelSelector.MatchLabels)+len(n.LabelSelector.MatchExpressions) == 0
}
<|im_end|>
<|im_start|>assistant
// IsWildcard returns true if the selector is a wildcard selector.<|im_end|>


Save the model locally.

In [36]:
trainer.save_model('local-qwen2.5-1.5B-code-adapter')

Push the fine-tuned model to Hugging Face's model hub for sharing and future use.

In [37]:
trainer.push_to_hub()

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 23.2kB / 37.0MB            

  ...de-adapter/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

  ...adapter/training_args.bin:   1%|1         |  89.0B / 5.97kB            

CommitInfo(commit_url='https://huggingface.co/Juanxxo/qwen2.5-1.5B-code-adapter/commit/68c22eb8098836fca9f225bfb2c9e71a5c4b5f5d', commit_message='End of training', commit_description='', oid='68c22eb8098836fca9f225bfb2c9e71a5c4b5f5d', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Juanxxo/qwen2.5-1.5B-code-adapter', endpoint='https://huggingface.co', repo_type='model', repo_id='Juanxxo/qwen2.5-1.5B-code-adapter'), pr_revision=None, pr_num=None)